# 01 Validate Repository

Executable repository and result-audit notebook. It checks source files, scripts, docs, notebooks, optional result directories, and writes machine-readable audit tables. Smoke outputs are reported but never treated as scientific evidence.

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wwgpt').exists():
            return p
    raise RuntimeError('Could not locate repository root')

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import (
    completed_runs, discover_experiment_runs, normalize_metrics, terminal_results,
    add_generalization_measures, vocab_size_from_artifacts, summary, align_curves,
    paired_curve_differences,
)

RESULTS_ROOT = Path(globals().get('RESULTS_ROOT', REPO_ROOT / 'results'))
ANALYSIS_DIR = Path(globals().get('ANALYSIS_DIR', RESULTS_ROOT / 'notebook_analysis'))
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Notebook outputs: {ANALYSIS_DIR}')


## Repository file audit

In [ ]:
expected = {
    'package': [REPO_ROOT/'src'/'wwgpt'/p for p in ['config.py','model.py','train.py','ww.py','analysis.py','scaling.py','data.py','cli.py']],
    'scripts': [REPO_ROOT/'scripts'/p for p in ['run_smoke_test.sh','run_five_seeds.sh','analyze_five_seeds.sh','run_scaling_grid.sh','analyze_scaling_grid.sh']],
    'docs': [REPO_ROOT/p for p in ['README.md','docs/METHODOLOGY.md','docs/RESULT_SCHEMA.md','docs/EXPERIMENT_PROTOCOL.md']],
    'notebooks': sorted((REPO_ROOT/'notebooks').glob('*.ipynb')),
}
rows=[]
for group, paths in expected.items():
    for path in paths:
        rows.append({'group': group, 'path': str(path.relative_to(REPO_ROOT)), 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0})
audit = pd.DataFrame(rows)
audit.to_csv(ANALYSIS_DIR/'repository_file_audit.csv', index=False)
display(audit)
missing = audit.loc[~audit['exists']]
if not missing.empty:
    raise AssertionError('Missing expected repository files: ' + ', '.join(missing['path']))


## Notebook substance audit

In [ ]:
nb_rows=[]
for path in sorted((REPO_ROOT/'notebooks').glob('*.ipynb')):
    data=json.loads(path.read_text())
    cells=data.get('cells', [])
    code_cells=[c for c in cells if c.get('cell_type')=='code']
    md_cells=[c for c in cells if c.get('cell_type')=='markdown']
    code_chars=sum(len(''.join(c.get('source', []))) for c in code_cells)
    nb_rows.append({'notebook': path.name, 'cells': len(cells), 'markdown_cells': len(md_cells), 'code_cells': len(code_cells), 'code_characters': code_chars, 'substantive': len(code_cells) >= 3 and code_chars >= 1200})
notebook_audit=pd.DataFrame(nb_rows)
notebook_audit.to_csv(ANALYSIS_DIR/'notebook_substance_audit.csv', index=False)
display(notebook_audit)
if not notebook_audit['substantive'].all():
    raise AssertionError('One or more notebooks appear to be placeholders')


## Result discovery audit

In [ ]:
runs = discover_experiment_runs(RESULTS_ROOT)
rows=[]
for r in runs:
    art=r['artifacts']; man=art.get('manifest.json', {})
    m=normalize_metrics(art.get('metrics.csv', pd.DataFrame()))
    rows.append({'pair_id': r['pair_id'], 'seed': r['seed'], 'optimizer': r['optimizer'], 'run_dir': str(r['run_dir']), 'selection_note': r['selection_note'], 'has_manifest': bool(man), 'has_metrics': not m.empty, 'valid_for_science': man.get('valid_for_science'), 'smoke_test': man.get('smoke_test'), 'metric_rows': len(m)})
run_audit=pd.DataFrame(rows)
run_audit.to_csv(ANALYSIS_DIR/'result_discovery_audit.csv', index=False)
if run_audit.empty:
    display(Markdown('No result runs found. Repository validation still completed; run training scripts to populate result audits.'))
else:
    display(run_audit)
    display(run_audit.groupby(['optimizer','valid_for_science','smoke_test'], dropna=False).size().reset_index(name='runs'))
